In [ ]:
import kagglehub
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter # Importation nécessaire

import seaborn as sns

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
# Download latest version
input_path = Path(
    kagglehub.dataset_download("mirichoi0218/insurance")
)
print("Path to dataset files:", input_path)

output_path = Path.cwd().parent / "outputs"
output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load the dataset
df = pd.read_csv(input_path / "insurance.csv")

## 1. Exploratory Data Analysis

In [ ]:
# Display the first few rows and column information
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
p75 = 16639.912515
p25 = 4740.287150
iqr = p75-p25
limite_sup_aberr = p75 + 1.5*iqr
limite_sup_aberr

In [ ]:
avg_charges = df["charges"].mean()

In [ ]:
# --- (Définition de la fonction de formatage) ---
def format_milliers(x, pos):
    return f"${x:,.0f}"#.replace(",", " ")
formateur_charges = FuncFormatter(format_milliers)

In [ ]:
# Set up the figure size
plt.figure(figsize=(15, 5))

# Subplot 1: Distribution of "charges"
plt.subplot(1, 2, 1)
sns.histplot(df["charges"], kde=True)
plt.axvline(x=avg_charges, color="red", linestyle='--', linewidth=2, label=f"Moyenne : ${avg_charges:,.0f}")
plt.title("Distribution des Frais Médicaux (charges)")
plt.xlabel("Frais Médicaux")
plt.ylabel("Fréquence")
plt.gca().xaxis.set_major_formatter(formateur_charges)
plt.xticks(rotation=45, ha='right')
plt.legend()

# Subplot 2: Distribution of "charges" / boxplot
plt.subplot(1, 2, 2)
sns.boxplot(y="charges", data=df)
plt.title("Boxplot des Frais Médicaux (charges)")
plt.gca().yaxis.set_major_formatter(formateur_charges)
plt.ylabel("Frais Médicaux")

plt.savefig(output_path / "eda_insurance_0.png")

In [ ]:
plt.figure(figsize=(20, 5))
# Subplot 1: "charges" vs "smoker" (Box plot)
plt.subplot(1, 3, 1)
sns.boxplot(x="smoker", y="charges", data=df)
plt.title("Charges vs Fumeur")
plt.xlabel("Fumeur (smoker)")
plt.ylabel("Frais Médicaux")
plt.gca().yaxis.set_major_formatter(formateur_charges)
# plt.xticks(rotation=45, ha='right')

# Subplot 2: "charges" vs "age" (Scatter plot)
plt.subplot(1, 3, 2)
sns.scatterplot(x="age", y="charges", hue="smoker", data=df)
plt.title("Charges vs Âge (par Fumeur)")
plt.xlabel("Âge")
plt.ylabel("Frais Médicaux")
plt.gca().yaxis.set_major_formatter(formateur_charges)

# Subplot 3: "charges" vs "bmi" (Scatter plot, colored by smoker)
plt.subplot(1, 3, 3)
sns.scatterplot(x="bmi", y="charges", hue="smoker", data=df)
plt.title("Charges vs IMC (par Fumeur)")
plt.xlabel("IMC (bmi)")
plt.ylabel("Frais Médicaux")
plt.gca().yaxis.set_major_formatter(formateur_charges)

plt.tight_layout()
plt.savefig(output_path / "eda_insurance_1.png")

 

In [ ]:
plt.figure(figsize=(20, 5))

# Subplot 2: "charges" vs "sex" (Box plot)
plt.subplot(1, 3, 1)
sns.boxplot(x="sex", y="charges", data=df)
plt.title("Charges vs Sex")
plt.xlabel("Sex")
plt.ylabel("Frais Médicaux")
plt.gca().yaxis.set_major_formatter(formateur_charges)

# Subplot 2: "charges" vs "children" (Box plot)
plt.subplot(1, 3, 2)
sns.boxplot(x="children", y="charges", data=df)
plt.title("Charges vs Nombre d\"Enfants")
plt.xlabel("Nombre d\"Enfants (children)")
plt.ylabel("Frais Médicaux")
plt.gca().yaxis.set_major_formatter(formateur_charges)

# Subplot 3: "charges" vs "region" (Box plot)
plt.subplot(1, 3, 3)
sns.boxplot(x="region", y="charges", data=df)
plt.title("Charges vs Region")
plt.ylabel("Frais Médicaux")
plt.gca().yaxis.set_major_formatter(formateur_charges)

plt.tight_layout()
plt.savefig(output_path / "eda_insurance_2.png")

In [ ]:
max_charges = max(df['charges'])
df.query(f"charges == {max_charges}")

## 2. Modeling

In [ ]:
# Define features and target
X = df.drop("charges", axis=1)
y = df["charges"]

In [ ]:
# Apply log transformation to the target variable to handle skewness
# Add a small constant (1) to handle potential zero values, though not strictly necessary here
y_log = np.log1p(y)

In [ ]:
# Identify categorical and numerical columns
categorical_features = X.select_dtypes(include=["object"]).columns
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns

In [ ]:
# Create the preprocessing pipelines for numerical and categorical features
numerical_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop='first'))
])

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
# --- Model Definition and Training ---
# Split the data
X_train, X_test, y_log_train, y_log_test = train_test_split(X, y_log, test_size=0.2, random_state=42)
y_test = np.expm1(y_log_test) # Keep the original test target for final evaluation
y_train = np.expm1(y_log_train)

In [ ]:
# Model 1: Linear Regression
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

lr_pipeline.fit(X_train, y_log_train)
lr_pred_log = lr_pipeline.predict(X_test)
lr_pred = np.expm1(lr_pred_log) # Inverse transform predictions

lr_pred_train_log = lr_pipeline.predict(X_train)
lr_pred_train = np.expm1(lr_pred_train_log) # Inverse transform predictions

In [ ]:
# Model 2: Random Forest Regressor
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_log_train)
rf_pred_log = rf_pipeline.predict(X_test)
rf_pred = np.expm1(rf_pred_log) # Inverse transform predictions

rf_pred_train_log = rf_pipeline.predict(X_train)
rf_pred_train = np.expm1(rf_pred_train_log) # Inverse transform predictions

In [ ]:
# --- Evaluation ---

# p = nombre de prédicteurs
p = lr_pipeline.named_steps['preprocessor'].transform(X_test).shape[1] 


# Function to calculate and print metrics
def evaluate_model(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    n = len(y_true)
    r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return rmse, r2, r2_adj

lr_rmse, lr_r2, lr_r2_adj = evaluate_model(y_test, lr_pred)
lr_rmse_train, lr_r2_train, lr_r2_adj_train = evaluate_model(y_train, lr_pred_train)

rf_rmse, rf_r2, rf_r2_adj = evaluate_model(y_test, rf_pred)
rf_rmse_train, rf_r2_train, rf_r2_adj_train = evaluate_model(y_train, rf_pred_train)

perf_model = pd.DataFrame(
    [
        {
            "Model": "Linear Regression",
            "RMSE - Train": lr_rmse_train,
            "R-squared - Train": lr_r2_train,
            "R-squared adjusted - Train": lr_r2_adj_train,
            "RMSE - Test": lr_rmse,
            "R-squared - Test": lr_r2,
            "R-squared adjusted - Test": lr_r2_adj,
        },
        {
            "Model": "Random Forest Regression",
            "RMSE - Train": rf_rmse_train,
            "R-squared - Train": rf_r2_train,
            "R-squared adjusted - Train": rf_r2_adj_train,
            "RMSE - Test": rf_rmse,
            "R-squared - Test": rf_r2,
            "R-squared adjusted - Test": rf_r2_adj,
        },
    ]
)
perf_model.to_csv(output_path / "Model performance.csv")

In [ ]:
# --- Feature Importance (from Random Forest) ---
# Get feature names after one-hot encoding
onehot_cols = (
    lr_pipeline
    .named_steps["preprocessor"]
    .named_transformers_["cat"]
    .named_steps["onehot"]
    .get_feature_names_out(categorical_features)
)
feature_names = list(numerical_features) + list(onehot_cols)

# Get importances from Random Forest
rf_importances = (
    rf_pipeline
    .named_steps["regressor"]
    .feature_importances_
)
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": rf_importances})
importance_df = importance_df.sort_values(by="Importance", ascending=False)
importance_df.to_csv(output_path / "Random forest - Feature importance.csv")